In [ ]:
from scipy.stats import f_oneway #for one-way ANOVA
from sklearn.decomposition import PCA #for PCA
from scipy.stats import t #to calculate 95% CI
import numpy as np
import ast
import pandas as pd
import os
import matplotlib.pyplot as plt
import math
import json
import pq_post_dda as pqpost
import ms_entropy as me 
import sqlite3
import tempfile
import ijson.backends.python as ijson_python
import sqlite3
import tempfile


In [ ]:
from decimal import Decimal

# This function is used to serialize numpy data types and other non-serializable objects to JSON format. 
def _json_serializer(obj):

    if isinstance(
        obj,
        (
            np.integer,
            np.int64,
            np.int32,
        )
    ):

        return int(obj)

    if isinstance(
        obj,
        (
            np.floating,
            np.float64,
            np.float32,
        )
    ):

        value = float(obj)

        if np.isfinite(value):

            return value

        return None

    if isinstance(
        obj,
        Decimal
    ):

        value = float(obj)

        if np.isfinite(value):

            return value

        return None

    if isinstance(
        obj,
        np.ndarray
    ):

        return obj.tolist()

    if obj is None:

        return None

    raise TypeError(
        f"Object of type "
        f"{type(obj).__name__} "
        f"is not JSON serializable"
    )

In [ ]:
# This function calculates additional statistics for clusters based on the provided annotation results. 
# It processes each cluster, collects normalized intensities, checks for blank occurrences, performs ANOVA, 
# #calculates pairwise log fold changes (LFC), and determines quality control (QC) for each group within the cluster. 
# The results are returned as a DataFrame with one row per group and cluster combination.
def _calculate_additional_statistics_for_clusters(
    annotation_results,
    blank=None,
    alpha=0.05,
    min_consensus_replicates=4,
    max_cv=30,
    min_similarity=0.70,
    min_intensity=100,
):

    results = []

    # ================================================================
    # Process one cluster at a time
    # ================================================================

    for cluster_id, cluster_df in annotation_results.groupby(
        "cluster_id",
        sort=False
    ):

        # ============================================================
        # 1. COLLECT ALL NORMALIZED INTENSITIES FOR THIS CLUSTER
        #
        # Each row contains the replicates for ONE biological group.
        #
        # Example:
        #
        # p1 + cluster 3291 -> p1 replicates
        # p2 + cluster 3291 -> p2 replicates
        # p3 + cluster 3291 -> p3 replicates
        #
        # We combine them here so ANOVA can compare:
        #
        # p1 vs p2 vs p3
        # ============================================================

        group_intensities = {}

        for _, row in cluster_df.iterrows():

            row_group = row.get("group")

            for replicate in (
                row.get("overall_replicates") or []
            ):

                # ----------------------------------------------------
                # Biological group
                # ----------------------------------------------------

                replicate_group = replicate.get(
                    "_group"
                )

                # Fallback to the dataframe row's group
                if replicate_group is None:
                    replicate_group = row_group

                if replicate_group is None:
                    continue

                # ----------------------------------------------------
                # Normalized intensity
                # ----------------------------------------------------

                normalization = (
                    replicate.get("normalization")
                    or {}
                )

                intensity = normalization.get(
                    "normalized_precursor_intensity"
                )

                # ----------------------------------------------------
                # Validate intensity
                # ----------------------------------------------------

                if intensity is None:
                    continue

                if intensity == "NA":
                    continue

                try:
                    intensity = float(intensity)

                except (TypeError, ValueError):
                    continue

                if not np.isfinite(intensity):
                    continue

                # ----------------------------------------------------
                # Store intensity by biological group
                # ----------------------------------------------------

                group_intensities.setdefault(
                    replicate_group,
                    []
                ).append(intensity)

        # ============================================================
        # 2. CHECK WHETHER THE CLUSTER OCCURS IN A BLANK
        # ============================================================

        cluster_found_in_blank = False

        for _, row in cluster_df.iterrows():

            blank_match_files = (
                row.get("blank_match_files")
                or []
            )

            if blank_match_files:
                cluster_found_in_blank = True
                break

        # ============================================================
        # 3. CALCULATE ANOVA ONCE FOR THE WHOLE CLUSTER
        #
        # IMPORTANT:
        # group_intensities contains ALL biological groups.
        # ============================================================

        anova_groups = [
            values
            for values in group_intensities.values()
            if len(values) >= 2
        ]

        anova_f_stat = np.nan
        anova_p_value = np.nan

        if (
            not cluster_found_in_blank
            and len(anova_groups) >= 2
        ):

            try:

                anova_f_stat, anova_p_value = (
                    f_oneway(
                        *anova_groups
                    )
                )

            except Exception as e:

                print(
                    f"ANOVA failed for "
                    f"cluster {cluster_id}: {e}"
                )

        # ============================================================
        # 4. CALCULATE PAIRWISE LFC ONCE FOR THE WHOLE CLUSTER
        # ============================================================

        pairwise_lfc = {}

        groups = sorted(
            group_intensities.keys()
        )

        for i in range(len(groups)):

            for j in range(
                i + 1,
                len(groups)
            ):

                group_i = groups[i]
                group_j = groups[j]

                values_i = group_intensities[
                    group_i
                ]

                values_j = group_intensities[
                    group_j
                ]

                # ----------------------------------------------------
                # Need values in both groups
                # ----------------------------------------------------

                if not values_i:
                    continue

                if not values_j:
                    continue

                mean_i = np.mean(
                    values_i
                )

                mean_j = np.mean(
                    values_j
                )

                # ----------------------------------------------------
                # Avoid division by zero
                # ----------------------------------------------------

                if (
                    not np.isfinite(mean_i)
                    or not np.isfinite(mean_j)
                ):
                    continue

                if (
                    mean_i <= 0
                    or mean_j <= 0
                ):
                    continue

                # ----------------------------------------------------
                # log2 fold change
                #
                # Positive:
                # group_i is higher than group_j
                #
                # Negative:
                # group_i is lower than group_j
                # ----------------------------------------------------

                lfc = np.log2(
                    mean_i / mean_j
                )

                pairwise_lfc[
                    f"{group_i}_vs_{group_j}"
                ] = lfc

        # ============================================================
        # 5. DETERMINE QC FOR EACH GROUP
        #
        # We calculate this separately for every group + cluster
        # combination.
        # ============================================================

        group_qc_results = {}

        for _, row in cluster_df.iterrows():

            row_group = row.get(
                "group"
            )

            annotation_name = row.get(
                "identified_annotation_name"
            )

            annotation_saturation = row.get(
                "identified_annotation_name"
            )

            is_unidentified = (
                annotation_name
                == "Unidentified"
            )

            passed_qc = True
            qc_reason = None

            # --------------------------------------------------------
            # Blank
            # --------------------------------------------------------

            blank_match_files = (
                row.get("blank_match_files")
                or []
            )

            if blank_match_files:

                passed_qc = False

                qc_reason = (
                    "found_in_blank"
                )

            # --------------------------------------------------------
            # Only continue QC if not found in blank
            # --------------------------------------------------------

            if passed_qc:

                cv = row.get(
                    "intensity_cv"
                )

                mean_intensity = row.get(
                    "mean_intensity"
                )

                # ====================================================
                # UNIDENTIFIED
                # ====================================================

                if is_unidentified:

                    ms2_replicates = (
                        row.get(
                            "MS2_replicates"
                        )
                        or []
                    )

                    # ------------------------------------------------
                    # Minimum MS2 replicates
                    # ------------------------------------------------

                    if len(
                        ms2_replicates
                    ) < 4:

                        passed_qc = False

                        qc_reason = (
                            "too_few_MS2_replicates"
                        )

                    # ------------------------------------------------
                    # CV
                    # ------------------------------------------------

                    elif (
                        cv is not None
                        and not pd.isna(cv)
                        and cv > max_cv
                    ):

                        passed_qc = False

                        qc_reason = (
                            "CV_above_threshold"
                        )

                    # ------------------------------------------------
                    # Intensity
                    # ------------------------------------------------

                    elif (
                        mean_intensity is not None
                        and not pd.isna(
                            mean_intensity
                        )
                        and mean_intensity
                        < min_intensity
                    ):

                        passed_qc = False

                        qc_reason = (
                            "mean_intensity_below_threshold"
                        )

                # ====================================================
                # IDENTIFIED
                # ====================================================

                else:

                    consensus_replicate_count = sum(
                        1
                        for replicate in (
                            row.get(
                                "overall_replicates"
                            )
                            or []
                        )
                        if replicate.get(
                            "replicate_annotation_name"
                        )
                        == annotation_name
                    )

                    # ------------------------------------------------
                    # Minimum consensus replicates
                    # ------------------------------------------------

                    if (
                        consensus_replicate_count
                        < min_consensus_replicates
                    ):

                        passed_qc = False

                        qc_reason = (
                            "too_few_consensus_replicates"
                        )

                    # ------------------------------------------------
                    # CV
                    # ------------------------------------------------

                    elif (
                        cv is not None
                        and not pd.isna(cv)
                        and cv > max_cv
                    ):

                        passed_qc = False

                        qc_reason = (
                            "CV_above_threshold"
                        )

                    # ------------------------------------------------
                    # Consensus entropy similarity
                    #
                    # This will be calculated below for this row.
                    # ------------------------------------------------

                    # ------------------------------------------------
                    # Mean intensity
                    # ------------------------------------------------

                    elif (
                        mean_intensity is not None
                        and not pd.isna(
                            mean_intensity
                        )
                        and mean_intensity
                        < min_intensity
                    ):

                        passed_qc = False

                        qc_reason = (
                            "mean_intensity_below_threshold"
                        )

            # --------------------------------------------------------
            # Store group-specific QC result
            # --------------------------------------------------------

            group_qc_results[
                row_group
            ] = {
                "passed_qc": passed_qc,
                "qc_reason": qc_reason,
            }

        # ============================================================
        # 6. NOW CREATE ONE RESULT FOR EACH
        #    GROUP + CLUSTER_ID COMBINATION
        # ============================================================

        for _, row in cluster_df.iterrows():

            group = row.get(
                "group"
            )

            # ========================================================
            # Consensus/conflicting MS2 statistics
            #
            # These belong to THIS group + cluster combination.
            # ========================================================

            consensus_annotation = row.get(
                "identified_annotation_name"
            )

            consensus_match_scores = []
            consensus_entropy_similarities = []

            conflicting_match_scores = []
            conflicting_entropy_similarities = []

            charges = []

            for replicate in (
                row.get(
                    "overall_replicates"
                )
                or []
            ):

                replicate_annotation = (
                    replicate.get(
                        "replicate_annotation_name"
                    )
                )

                msp_match = (
                    replicate.get(
                        "MSP_match"
                    )
                    or {}
                )

                match = (
                    msp_match.get(
                        "match"
                    )
                    or {}
                )

                match_score = match.get(
                    "score"
                )

                entropy_similarity = (
                    msp_match.get(
                        "MSP_entropy_similarity"
                    )
                )

                charge = replicate.get("charge_state")

                if isinstance(
                    charge,
                    (int, float, np.number)
                ):
                
                    charges.append(float(charge))

                    avg_charge = np.mean(charges)

                # ----------------------------------------------------
                # Consensus
                # ----------------------------------------------------

                if (
                    replicate_annotation
                    == consensus_annotation
                ):

                    if isinstance(
                        match_score,
                        (
                            int,
                            float,
                            np.number
                        )
                    ):

                        if np.isfinite(
                            float(match_score)
                        ):

                            consensus_match_scores.append(
                                float(match_score)
                            )

                    if isinstance(
                        entropy_similarity,
                        (
                            int,
                            float,
                            np.number
                        )
                    ):

                        if np.isfinite(
                            float(
                                entropy_similarity
                            )
                        ):

                            consensus_entropy_similarities.append(
                                float(
                                    entropy_similarity
                                )
                            )

                # ----------------------------------------------------
                # Conflicting
                # ----------------------------------------------------

                else:

                    if isinstance(
                        match_score,
                        (
                            int,
                            float,
                            np.number
                        )
                    ):

                        if np.isfinite(
                            float(match_score)
                        ):

                            conflicting_match_scores.append(
                                float(match_score)
                            )

                    if isinstance(
                        entropy_similarity,
                        (
                            int,
                            float,
                            np.number
                        )
                    ):

                        if np.isfinite(
                            float(
                                entropy_similarity
                            )
                        ):

                            conflicting_entropy_similarities.append(
                                float(
                                    entropy_similarity
                                )
                            )

            # ========================================================
            # Calculate averages for THIS group + cluster
            # ========================================================

            if consensus_match_scores:

                avg_consensus_match_score = (
                    np.mean(
                        consensus_match_scores
                    )
                )

            else:

                avg_consensus_match_score = np.nan

            if consensus_entropy_similarities:

                avg_consensus_entropy_similarity = (
                    np.mean(
                        consensus_entropy_similarities
                    )
                )

            else:

                avg_consensus_entropy_similarity = (
                    np.nan
                )

            if conflicting_match_scores:

                avg_conflicting_match_score = (
                    np.mean(
                        conflicting_match_scores
                    )
                )

            else:

                avg_conflicting_match_score = (
                    np.nan
                )

            if conflicting_entropy_similarities:

                avg_conflicting_entropy_similarity = (
                    np.mean(
                        conflicting_entropy_similarities
                    )
                )

            else:

                avg_conflicting_entropy_similarity = (
                    np.nan
                )

            # ========================================================
            # Apply entropy similarity QC for identified features
            #
            # This has to happen here because the average similarity
            # belongs to this specific group + cluster row.
            # ========================================================

            qc_info = group_qc_results.get(
                group,
                {
                    "passed_qc": False,
                    "qc_reason": "group_not_found",
                }
            )

            passed_qc = qc_info[
                "passed_qc"
            ]

            qc_reason = qc_info[
                "qc_reason"
            ]

            if passed_qc:

                is_unidentified = (
                    consensus_annotation
                    == "Unidentified"
                )

                if not is_unidentified:

                    if (
                        np.isnan(
                            avg_consensus_entropy_similarity
                        )
                    ):

                        passed_qc = False

                        qc_reason = (
                            "no_consensus_entropy_similarity"
                        )

                    elif (
                        avg_consensus_entropy_similarity
                        < min_similarity
                    ):

                        passed_qc = False

                        qc_reason = (
                            "entropy_similarity_below_threshold"
                        )

            # ========================================================
            # Create result from THIS ROW
            #
            # This is what ensures one result per
            # group + cluster_id.
            # ========================================================

            result = row.to_dict()

            # --------------------------------------------------------
            # Make sure these fields exist
            # --------------------------------------------------------
            result["charge"] = (avg_charge)

            result["annotation_saturation"] = (annotation_saturation)

            result[
                "avg_consensus_entropy_similarity"
            ] = (
                avg_consensus_entropy_similarity
            )

            result[
                "avg_consensus_match_score"
            ] = (
                avg_consensus_match_score
            )

            result[
                "avg_conflicting_entropy_similarity"
            ] = (
                avg_conflicting_entropy_similarity
            )

            result[
                "avg_conflicting_match_score"
            ] = (
                avg_conflicting_match_score
            )

            # ========================================================
            # QC result
            # ========================================================

            result[
                "statistics_passed_qc"
            ] = passed_qc

            result[
                "statistics_qc_reason"
            ] = qc_reason

            # ========================================================
            # Cluster-level ANOVA
            #
            # Same ANOVA is attached to every group row belonging
            # to this cluster.
            # ========================================================

            if (
                passed_qc
                and not cluster_found_in_blank
            ):

                result[
                    "anova_f_stat"
                ] = anova_f_stat

                result[
                    "anova_p_value"
                ] = anova_p_value

                result[
                    "pairwise_lfc"
                ] = pairwise_lfc

            else:

                result[
                    "anova_f_stat"
                ] = np.nan

                result[
                    "anova_p_value"
                ] = np.nan

                result[
                    "pairwise_lfc"
                ] = {}

            # ========================================================
            # Store THIS group's normalized intensities
            #
            # Not all groups.
            # ========================================================

            result[
                "group_intensities"
            ] = {
                group: group_intensities.get(
                    group,
                    []
                )
            }

            # ========================================================
            # Optional: store all cluster-level group intensities
            #
            # This is useful for checking ANOVA/LFC.
            # ========================================================

            result[
                "anova_group_intensities"
            ] = group_intensities

            # ========================================================
            # Add result
            # ========================================================

            results.append(
                result
            )

    # ================================================================
    # Return one row per group + cluster_id
    # ================================================================

    return pd.DataFrame(results)    

In [ ]:
def get_numeric_values(values):

    numeric_values = []

    for value in values:

        if value is None:
            continue

        if value == "No MS2":
            continue

        try:
            numeric_values.append(
                float(value)
            )

        except (TypeError, ValueError):
            continue

    return numeric_values

# ============================================================
# Extract normalized intensity
# ============================================================

def get_normalized_intensity(cluster):
    """
    Get the normalized precursor intensity from one
    replicate-level cluster dictionary.
    """

    normalization = cluster.get("normalization")

    if not normalization:
        return None

    intensity = normalization.get(
        "normalized_precursor_intensity"
    )

    if intensity is None:
        return None

    if intensity == "NA":
        return None

    try:
        return float(intensity)

    except (TypeError, ValueError):
        return None

def get_blank_match_details(cleaned_result, blank_file_id):

    for replicate in cleaned_result.get(
        "overall_replicates",
        []
    ):

        if replicate.get("file_id") != blank_file_id:
            continue

        details = replicate.get(
            "blank_match_details"
        )

        if details and details.get("blank_match"):
            return details

    return None

def calculate_additional_statistics_large_json(
    input_file,
    output_file,
    blank=None,
    alpha=0.05,
    min_consensus_replicates=4,
    max_cv=30,
    min_similarity=0.70,
    min_intensity=100,
    min_replicates=4,
):
    """
    Process a very large annotation-results JSON file without
    loading the entire file into memory.

    Expected JSON structure:

    [
        {
            "group": "b",
            "cluster_id": 115,
            ...
        },
        {
            "group": "b",
            "cluster_id": 116,
            ...
        }
    ]

    The JSON file is a top-level array of result dictionaries.

    OPTION A FILTERING
    ------------------

    Filtering occurs at the individual group + cluster_id level.

    A row is skipped BEFORE statistical calculations if:

        mean_intensity < min_intensity

    OR:

        n_replicates < min_replicates

    Other group + cluster_id rows from the same cluster are retained.

    NaN, Infinity and -Infinity in the source JSON are automatically
    converted to None while streaming.

    One cluster at a time is loaded into pandas.
    """

    # ================================================================
    # Create temporary SQLite database
    # ================================================================

    with tempfile.NamedTemporaryFile(
        suffix=".db",
        delete=False
    ) as temp_file:

        database_path = temp_file.name

    conn = sqlite3.connect(
        database_path
    )

    cursor = conn.cursor()

    # ================================================================
    # Create table
    # ================================================================

    cursor.execute(
        """
        CREATE TABLE results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            cluster_id TEXT,
            row_json TEXT
        )
        """
    )

    cursor.execute(
        """
        CREATE INDEX idx_cluster_id
        ON results(cluster_id)
        """
    )

    conn.commit()

    # ================================================================
    # Counters
    # ================================================================

    total_rows = 0
    retained_rows = 0

    skipped_low_intensity = 0
    skipped_low_replicates = 0
    skipped_invalid = 0

    print()
    print("Reading large JSON file...")
    print()

    # ================================================================
    # PASS 1
    #
    # Stream JSON and filter rows.
    #
    # The Python ijson backend is used because the source JSON
    # contains NaN / Infinity / -Infinity values.
    # ================================================================

    total_rows = 0
    retained_rows = 0

    skipped_low_intensity = 0
    skipped_low_replicates = 0
    skipped_invalid = 0

    print()
    print("Reading large JSON file...")
    print()


    with open(input_file, "rb") as infile:

        for row in ijson_python.items(
            infile,
            "item"
        ):

            total_rows += 1

            # ========================================================
            # Get filtering values
            # ========================================================

            mean_intensity = row.get(
                "mean_intensity"
            )

            n_replicates = row.get(
                "n_replicates"
            )

            # ========================================================
            # Missing intensity / NaN
            # ========================================================

            if mean_intensity is None:

                skipped_invalid += 1

                continue

            # ========================================================
            # Missing replicate count
            # ========================================================

            if n_replicates is None:

                skipped_invalid += 1

                continue

            # ========================================================
            # Convert intensity
            # ========================================================

            try:

                mean_intensity = float(
                    mean_intensity
                )

            except (
                TypeError,
                ValueError
            ):

                skipped_invalid += 1

                continue

            # ========================================================
            # Convert replicate count
            # ========================================================

            try:

                n_replicates = int(
                    n_replicates
                )

            except (
                TypeError,
                ValueError
            ):

                skipped_invalid += 1

                continue

            # ========================================================
            # Catch NaN / Infinity / -Infinity
            # ========================================================

            if not np.isfinite(
                mean_intensity
            ):

                skipped_invalid += 1

                continue

            # ========================================================
            # OPTION A FILTERING
            #
            # Filter ONLY this group + cluster row.
            #
            # The entire cluster is NOT removed.
            # ========================================================

            if mean_intensity < min_intensity:

                skipped_low_intensity += 1

                continue

            if n_replicates < min_replicates:

                skipped_low_replicates += 1

                continue

            # ========================================================
            # Get cluster ID
            # ========================================================

            cluster_id = row.get(
                "cluster_id"
            )

            if cluster_id is None:

                skipped_invalid += 1

                continue

            # ========================================================
            # Store surviving row
            # ========================================================

            cursor.execute(
                """
                INSERT INTO results (
                    cluster_id,
                    row_json
                )
                VALUES (?, ?)
                """,
                (
                    str(cluster_id),
                    json.dumps(
                        row,
                        ensure_ascii=False,
                        allow_nan=True,
                        default=_json_serializer
                    )
                )
            )

            retained_rows += 1

            # ========================================================
            # Commit periodically
            # ========================================================

            if retained_rows % 10_000 == 0:
                print(
                    f"Reading rows 10% complete"
                )

                conn.commit()

            # ========================================================
            # Progress
            # ========================================================

            if total_rows % 100_000 == 0:

                print(
                    f"Reading rows 100% complete"
                    f"Read {total_rows:,} rows | "
                    f"Retained {retained_rows:,} | "
                    f"Skipped intensity "
                    f"{skipped_low_intensity:,} | "
                    f"Skipped replicates "
                    f"{skipped_low_replicates:,}"
                )
    conn.commit()

    # ================================================================
    # First-pass summary
    # ================================================================

    print()
    print("=" * 70)
    print("FIRST PASS COMPLETE")
    print("=" * 70)

    print(
        f"Total rows:             {total_rows:,}"
    )

    print(
        f"Retained rows:          {retained_rows:,}"
    )

    print(
        f"Skipped low intensity:  "
        f"{skipped_low_intensity:,}"
    )

    print(
        f"Skipped low replicates: "
        f"{skipped_low_replicates:,}"
    )

    print(
        f"Skipped invalid:        "
        f"{skipped_invalid:,}"
    )

    print()

    # ================================================================
    # Number of clusters
    # ================================================================

    cursor.execute(
        """
        SELECT COUNT(DISTINCT cluster_id)
        FROM results
        """
    )

    n_clusters = cursor.fetchone()[0]

    print(
        f"Clusters to process:    {n_clusters:,}"
    )

    print()

    # ================================================================
    # Open output JSON
    # ================================================================

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as outfile:

        outfile.write("[")

        first_result = True

        # ============================================================
        # PASS 2
        #
        # Process one cluster at a time.
        # ============================================================

        cursor.execute(
            """
            SELECT DISTINCT cluster_id
            FROM results
            ORDER BY id
            """
        )

        cluster_number = 0

        for cluster_row in cursor:

            cluster_id = cluster_row[0]

            cluster_number += 1

            # ========================================================
            # Retrieve this cluster only
            # ========================================================

            cursor2 = conn.cursor()

            cursor2.execute(
                """
                SELECT row_json
                FROM results
                WHERE cluster_id = ?
                """,
                (cluster_id,)
            )

            cluster_records = [
                json.loads(
                    row[0]
                )
                for row in cursor2.fetchall()
            ]

            cursor2.close()

            # ========================================================
            # Convert to DataFrame
            # ========================================================

            cluster_df = pd.DataFrame(
                cluster_records
            )

            # ========================================================
            # Calculate statistics
            #
            # This is your existing statistical function.
            # ========================================================

            cluster_results = (
                _calculate_additional_statistics_for_clusters(
                    cluster_df,
                    blank=blank,
                    alpha=alpha,
                    min_consensus_replicates=(
                        min_consensus_replicates
                    ),
                    max_cv=max_cv,
                    min_similarity=min_similarity,
                    min_intensity=min_intensity,
                )
            )

            # ========================================================
            # Write results immediately
            # ========================================================

            for result in cluster_results.to_dict(
                orient="records"
            ):

                if not first_result:

                    outfile.write(",")

                outfile.write(
                    json.dumps(
                        result,
                        ensure_ascii=False,
                        default=_json_serializer
                    )
                )

                first_result = False

            # ========================================================
            # Explicitly release memory
            # ========================================================

            del cluster_records
            del cluster_df
            del cluster_results

            # ========================================================
            # Progress
            # ========================================================

            if (
                cluster_number % 100 == 0
                or cluster_number == n_clusters
            ):

                print(
                    f"Processed cluster "
                    f"{cluster_number:,} / "
                    f"{n_clusters:,}"
                )

        # ============================================================
        # Finish JSON
        # ============================================================

        outfile.write("]")

    # ================================================================
    # Close SQLite
    # ================================================================

    conn.close()

    # ================================================================
    # Final message
    # ================================================================

    print()
    print("=" * 70)
    print("PROCESSING COMPLETE")
    print("=" * 70)

    print(
        f"Output file:\n{output_file}"
    )

    print()

In [ ]:
calculate_additional_statistics_large_json(
    input_file=r"C:\Users\User\Annotation_Results.json",
    output_file="aAnnotation_Results_with_statistics.json",
    blank="b1",
    min_intensity=100,
    min_replicates=4,
    min_consensus_replicates=4,
    max_cv=30,
    min_similarity=0.75,
)

In [ ]:
def load_annotation_results(json_path):

    with open(json_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    return pd.DataFrame(records)

In [ ]:
# ============================================================
# Extract normalized intensity
# ============================================================

def get_normalized_intensity(cluster):
    """
    Get the normalized precursor intensity from one
    replicate-level cluster dictionary.
    """

    normalization = cluster.get("normalization")

    if not normalization:
        return None

    intensity = normalization.get(
        "normalized_precursor_intensity"
    )

    if intensity is None:
        return None

    if intensity == "NA":
        return None

    try:
        return float(intensity)

    except (TypeError, ValueError):
        return None

In [ ]:
annotation_results = pd.read_json(r"C:\Users\User\Annotation_Results_with_statistics.json")

In [ ]:
def _mean_ci_text(values, decimals=4):
    """Return mean +/- 95% CI as text, or NA when unavailable."""

    numeric_values = []

    for value in values:
        try:
            numeric_value = float(value)
        except (TypeError, ValueError):
            continue

        if np.isfinite(numeric_value):
            numeric_values.append(numeric_value)

    if not numeric_values:
        return "NA"

    mean_value = np.mean(numeric_values)

    if len(numeric_values) < 2:
        return f"{mean_value:.{decimals}f} +/- NA"

    standard_deviation = np.std(numeric_values, ddof=1)
    margin = t.ppf(0.975, len(numeric_values) - 1) * (
        standard_deviation / np.sqrt(len(numeric_values))
    )

    return f"{mean_value:.{decimals}f} +/- {margin:.{decimals}f}"


def make_significant_cluster_table(
    annotation_results_neg,
    alpha=0.05,
    mz_decimals=6,
    rt_decimals=3,
    entropy_decimals=3,
):
    """Create one summary row per statistically significant cluster."""


    statistics_df = pd.DataFrame(annotation_results_neg).copy()

    significant = statistics_df[
        pd.to_numeric(
            statistics_df["anova_p_value"],
            errors="coerce",
        ) < alpha
    ]

    comparison_names = sorted({
        comparison
        for pairwise_values in significant["pairwise_lfc"]
        for comparison in (pairwise_values or {})
        if "blanksolvextr" not in comparison.lower()
    })

    summary_rows = []

    for cluster_id, cluster_df in significant.groupby(
        "cluster_id",
        sort=False,
    ):
        first_row = cluster_df.iloc[0]
        replicates = []

        for value in cluster_df.get("overall_replicates", []):
            replicates.extend(value or [])

        mz_values = [
            replicate.get("precursor_mz")
            for replicate in replicates
        ]
        rt_values = [
            replicate.get("precursor_rt")
            for replicate in replicates
        ]
        entropy_values = [
            (replicate.get("MSP_match") or {}).get(
                "MSP_entropy_similarity"
            )
            for replicate in replicates
        ]

        row = {
            "Cluster_id": cluster_id,
            "Charge": float(first_row["charge"]),
            "Metabolite identity": first_row.get(
                "identified_annotation_name",
                "Unidentified",
            ),
            "Saturation": first_row.get(
                "identified_annotation_saturation"
            ),
            "average precursor m/z +/- 95% CI": _mean_ci_text(
                mz_values,
                decimals=mz_decimals,
            ),
            "average precursor RT +/- 95% CI": _mean_ci_text(
                rt_values,
                decimals=rt_decimals,
            ),
            "average entropy similarity +/- 95% CI": _mean_ci_text(
                entropy_values,
                decimals=entropy_decimals,
            ),
            "p_value": float(first_row["anova_p_value"]),
        }

        pairwise_values = first_row.get("pairwise_lfc") or {}

        for comparison in comparison_names:
            row[f"lfc {comparison}"] = pairwise_values.get(
                comparison,
                "NA",
            )

        summary_rows.append(row)

    return pd.DataFrame(
        summary_rows,
        columns=[
            "Cluster_id",
            "Charge",
            "Metabolite identity",
            "Saturation",
            "average precursor m/z +/- 95% CI",
            "average precursor RT +/- 95% CI",
            "average entropy similarity +/- 95% CI",
            "p_value",
            *[f"lfc {comparison}" for comparison in comparison_names],
        ],
    )


significant_cluster_table = make_significant_cluster_table(
    annotation_results,
    alpha=0.05,
)

significant_cluster_table

In [ ]:
significant_cluster_table.to_csv(
    "significant_cluster_table.csv",
    index=False,
    na_rep="NA",
)

print(
    "Saved significant cluster table to "
    "significant_cluster_table.csv"
)

In [ ]:
# Select identified metabolites with significant ANOVA results
top25_metabolites = significant_cluster_table.copy()

top25_metabolites["p_value"] = pd.to_numeric(
    top25_metabolites["p_value"],
    errors="coerce"
)

top25_metabolites = top25_metabolites[
    top25_metabolites["Metabolite identity"].notna()
    & ~top25_metabolites["Metabolite identity"]
        .str.contains("Unidentified", case=False, na=False)
    & top25_metabolites["p_value"].notna()
    & (top25_metabolites["p_value"] < 0.05)
]

# Smallest ANOVA p-values = most statistically altered
top25_metabolites = (
    top25_metabolites
    .sort_values("p_value")
    .head(25)
    .reset_index(drop=True)
)

display(top25_metabolites)

top25_metabolites.to_csv(
    "top25_significantly_altered_identified_metabolites.csv",
    index=False
)

In [ ]:
# Recover normalized group intensities for the selected clusters
statistics_df = significant_cluster_table.copy()

heatmap_rows = []

for _, selected_row in top25_metabolites.iterrows():
    cluster_id = selected_row["Cluster_id"]

    matching_rows = statistics_df[
        statistics_df["cluster_id"] == cluster_id
    ]

    if matching_rows.empty:
        continue

    intensity_dict = matching_rows.iloc[0].get(
        "anova_group_intensities",
        {}
    )

    heatmap_row = {
        "Metabolite": (
            f"{selected_row['Metabolite identity']} "
            f"[{cluster_id}]"
        )
    }

    for group, values in (intensity_dict or {}).items():
        if "blanksolvextr" in str(group).lower():
            continue

        numeric_values = pd.to_numeric(
            pd.Series(values),
            errors="coerce"
        ).dropna()

        if not numeric_values.empty:
            heatmap_row[group] = numeric_values.mean()

    heatmap_rows.append(heatmap_row)

heatmap_table = (
    pd.DataFrame(heatmap_rows)
    .set_index("Metabolite")
)

# Log transform and standardize each metabolite across groups
log_heatmap = np.log2(heatmap_table + 1)

heatmap_z = log_heatmap.sub(
    log_heatmap.mean(axis=1),
    axis=0
).div(
    log_heatmap.std(axis=1).replace(0, np.nan),
    axis=0
)

# Edit only the values on the right to rename groups on the plot
# without changing the underlying dataframe column names.
group_name_map = {
    "TOV": "A2780",
    "p1": "Plasma",
    "OVS": "OVCA429 WT + Sim",
    "OVKS": "OVCA429 KO + Sim",
    "OVKO": "OVCA429 KO",
    "OVC": "OVCA429 WT",
    "OV": "OVCAR3",
    "Kar": "Kuramochi",
}

plot_group_names = [
    group_name_map.get(group, group)
    for group in heatmap_z.columns
]

# Plot heatmap using matplotlib
fig, ax = plt.subplots(figsize=(12, 10))

image = ax.imshow(
    heatmap_z,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-2,
    vmax=2
)

ax.set_xticks(range(len(heatmap_z.columns)))
ax.set_xticklabels(
    plot_group_names,
    rotation=45,
    ha="right"
)

ax.set_yticks(range(len(heatmap_z.index)))
ax.set_yticklabels(heatmap_z.index)

ax.set_xlabel("Experimental group")
ax.set_ylabel("Metabolite")
ax.set_title("Top 25 significantly altered identified metabolites")

fig.colorbar(
    image,
    ax=ax,
    label="Row z-score of log2 normalized intensity"
)

plt.tight_layout()
plt.show()

In [ ]:
# THIS IS THE ONE WITHOUT PLASMA
# Recover normalized group intensities for the selected clusters
statistics_df = annotation_results.copy()

heatmap_rows = []

for _, selected_row in top25_metabolites.iterrows():
    cluster_id = selected_row["Cluster_id"]

    matching_rows = statistics_df[
        statistics_df["cluster_id"] == cluster_id
    ]

    if matching_rows.empty:
        continue

    intensity_dict = matching_rows.iloc[0].get(
        "anova_group_intensities",
        {}
    )

    heatmap_row = {
        "Metabolite": (
            f"{selected_row['Metabolite identity']}"
            + (
                f" | {selected_row['Saturation']}"
                if pd.notna(selected_row["Saturation"])
                else ""
            )
            + f" [{cluster_id}]"
        )
    }

    for group, values in (intensity_dict or {}).items():
        if "blanksolvextr" in str(group).lower():
            continue

        if "p1" in str(group).lower():
            continue

        numeric_values = pd.to_numeric(
            pd.Series(values),
            errors="coerce"
        ).dropna()

        if not numeric_values.empty:
            heatmap_row[group] = numeric_values.mean()

    heatmap_rows.append(heatmap_row)

heatmap_table = (
    pd.DataFrame(heatmap_rows)
    .set_index("Metabolite")
)

# Log transform and standardize each metabolite across groups
log_heatmap = np.log2(heatmap_table + 1)

heatmap_z = log_heatmap.sub(
    log_heatmap.mean(axis=1),
    axis=0
).div(
    log_heatmap.std(axis=1).replace(0, np.nan),
    axis=0
)

# Edit only the values on the right to rename groups on the plot
# without changing the underlying dataframe column names.
group_name_map = {
    "TOV": "A2780",
    "p1": "Plasma",
    "OVS": "OVCA429 WT + Sim",
    "OVKS": "OVCA429 KO + Sim",
    "OVKO": "OVCA429 KO",
    "OVC": "OVCA429 WT",
    "OV": "OVCAR3",
    "Kar": "Kuramochi",
}

plot_group_names = [
    group_name_map.get(group, group)
    for group in heatmap_z.columns
]

# Plot heatmap using matplotlib
fig, ax = plt.subplots(figsize=(12, 10))

image = ax.imshow(
    heatmap_z,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-2,
    vmax=2
)

ax.set_xticks(range(len(heatmap_z.columns)))
ax.set_xticklabels(
    plot_group_names,
    rotation=45,
    ha="right"
)

ax.set_yticks(range(len(heatmap_z.index)))
ax.set_yticklabels(heatmap_z.index)

ax.set_xlabel("Experimental group")
ax.set_ylabel("Metabolite")
ax.set_title("Top 25 significantly altered identified metabolites")

fig.colorbar(
    image,
    ax=ax,
    label="Row z-score of log2 normalized intensity"
)

plt.tight_layout()
plt.show()

In [ ]:
# Build a replicate-level intensity table for the selected metabolites
source_df = annotation_results.copy()
selected_cluster_ids = set(top25_metabolites["Cluster_id"])

replicate_rows = []
replicate_columns = []

for _, selected_row in top25_metabolites.iterrows():
    cluster_id = selected_row["Cluster_id"]
    cluster_rows = source_df[
        source_df["cluster_id"] == cluster_id
    ]

    metabolite_label = (
        f"{selected_row['Metabolite identity']} "
        f"[{cluster_id}]"
    )
    metabolite_values = {}
    group_counts = {}

    for _, cluster_row in cluster_rows.iterrows():
        row_group = cluster_row.get("group")

        for replicate in (cluster_row.get("overall_replicates") or []):
            group = replicate.get("_group") or row_group

            if group is None:
                continue

            if "blanksolvextr" in str(group).lower():
                continue

            normalization = replicate.get("normalization") or {}
            intensity = normalization.get(
                "normalized_precursor_intensity"
            )

            try:
                intensity = float(intensity)
            except (TypeError, ValueError):
                continue

            if not np.isfinite(intensity):
                continue

            group_counts[group] = group_counts.get(group, 0) + 1
            replicate_label = f"{group}_{group_counts[group]}"
            metabolite_values[replicate_label] = intensity

            if replicate_label not in replicate_columns:
                replicate_columns.append(replicate_label)

    replicate_rows.append({
        "Metabolite": metabolite_label,
        **metabolite_values,
    })

replicate_heatmap_table = (
    pd.DataFrame(replicate_rows)
    .set_index("Metabolite")
    .reindex(columns=replicate_columns)
)

replicate_heatmap_table.to_csv(
    "Annotations/top25_metabolites_replicate_intensities.csv",
    na_rep="NA",
)

display(replicate_heatmap_table)

# Log transform and standardize each metabolite across replicates
log_replicate_heatmap = np.log2(
    replicate_heatmap_table + 1
)

replicate_heatmap_z = log_replicate_heatmap.sub(
    log_replicate_heatmap.mean(axis=1),
    axis=0
).div(
    log_replicate_heatmap.std(axis=1).replace(0, np.nan),
    axis=0
)

fig, ax = plt.subplots(figsize=(16, 10))

image = ax.imshow(
    replicate_heatmap_z,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
)

ax.set_xticks(range(len(replicate_heatmap_z.columns)))
ax.set_xticklabels(
    replicate_heatmap_z.columns,
    rotation=90,
)

ax.set_yticks(range(len(replicate_heatmap_z.index)))
ax.set_yticklabels(replicate_heatmap_z.index)

ax.set_xlabel("Biological replicate")
ax.set_ylabel("Metabolite")
ax.set_title(
    "Top 25 significantly altered identified metabolites "
    "at replicate level"
)

fig.colorbar(
    image,
    ax=ax,
    label="Row z-score of log2 normalized intensity",
)

plt.tight_layout()
plt.show()

In [ ]:
# PCA using the whole identified dataset
# Each row becomes a sample and each column becomes an identified metabolite.
statistics_df = pd.DataFrame(
    annotation_results
).copy()

all_dataset_rows = statistics_df[
    statistics_df["identified_annotation_name"].notna()
    & ~statistics_df["identified_annotation_name"].str.contains(
        "Unidentified",
        case=False,
        na=False,
    )
    & ~statistics_df["group"].astype(str).str.lower().isin(
        ["b", "blank", "blanksolvextr", "p1_"]
    )
].copy()

sample_values = {}
sample_groups = {}

for _, statistics_row in all_dataset_rows.iterrows():
    cluster_id = statistics_row["cluster_id"]
    intensity_dict = statistics_row.get("group_intensities") or {}

    for group, values in intensity_dict.items():
        if str(group).lower() in {
            "b",
            "blank",
            "blanksolvextr",
            "p1_"
        }:
            continue

        for replicate_number, intensity in enumerate(values, start=1):
            try:
                intensity = float(intensity)
            except (TypeError, ValueError):
                continue

            if not np.isfinite(intensity):
                continue

            sample_id = f"{group}_{replicate_number}"
            sample_groups[sample_id] = group
            sample_values.setdefault(sample_id, {})[cluster_id] = intensity

whole_dataset_table = pd.DataFrame.from_dict(
    sample_values,
    orient="index",
)

# Retain features detected in at least 80% of samples. This reduces
# PCA distortion caused by features that are mostly missing.
minimum_detection_fraction = 0.80
feature_detection_fraction = whole_dataset_table.notna().mean(axis=0)
whole_dataset_table = whole_dataset_table.loc[
    :,
    feature_detection_fraction >= minimum_detection_fraction,
]

whole_dataset_table = whole_dataset_table.dropna(
    axis=0,
    how="all",
)

# Log-transform and impute missing values using each metabolite's median.
whole_dataset_log = np.log2(whole_dataset_table + 1)
whole_dataset_log = whole_dataset_log.apply(
    lambda column: column.fillna(column.median()),
    axis=0,
)

# Standardize metabolites before PCA.
whole_dataset_scaled = (
    whole_dataset_log - whole_dataset_log.mean(axis=0)
) / whole_dataset_log.std(axis=0).replace(0, np.nan)
whole_dataset_scaled = whole_dataset_scaled.fillna(0)

pca_whole_dataset = PCA(n_components=2)
whole_dataset_coordinates = pca_whole_dataset.fit_transform(
    whole_dataset_scaled
)

whole_dataset_pca_results = pd.DataFrame(
    whole_dataset_coordinates,
    index=whole_dataset_scaled.index,
    columns=["PC1", "PC2"],
)
whole_dataset_pca_results["Group"] = [
    sample_groups[sample_id]
    for sample_id in whole_dataset_pca_results.index
]

explained_variance = (
    pca_whole_dataset.explained_variance_ratio_ * 100
)

# Edit these values to customise the legend labels.
pca_group_name_map = {
    "TOV": "A2780",
    "p1_": "Plasma",
    "OVS": "OVCA429 WT + Sim",
    "OVKS": "OVCA429 KO + Sim",
    "OVKO": "OVCA429 KO",
    "OVC": "OVCA429 WT",
    "OV": "OVCAR3",
    "Kar": "Kuramochi",
}

whole_dataset_pca_results["Group label"] = (
    whole_dataset_pca_results["Group"]
    .map(pca_group_name_map)
    .fillna(whole_dataset_pca_results["Group"])
)

whole_dataset_table.to_csv(
    "whole_identified_dataset_sample_by_cluster.csv",
    na_rep="NA",
)
whole_dataset_pca_results.to_csv(
    "whole_identified_dataset_pca_coordinates.csv",
    index_label="Sample",
)

print(
    f"PCA input: {whole_dataset_table.shape[0]} samples "
    f"and {whole_dataset_table.shape[1]} identified metabolites."
)
print(
    f"PC1 explains {explained_variance[0]:.1f}% of variance; "
    f"PC2 explains {explained_variance[1]:.1f}%."
)

display(whole_dataset_pca_results)

fig, ax = plt.subplots(figsize=(11, 8))

for group_label, group_data in whole_dataset_pca_results.groupby(
    "Group label"
):
    ax.scatter(
        group_data["PC1"],
        group_data["PC2"],
        label=group_label,
        s=70,
        alpha=0.85,
    )

for sample_name, sample_data in whole_dataset_pca_results.iterrows():
    ax.annotate(
        sample_name,
        (sample_data["PC1"], sample_data["PC2"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=7,
    )

ax.set_xlabel(f"PC1 ({explained_variance[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({explained_variance[1]:.1f}%)")
ax.set_title("PCA of all identified metabolites")
ax.axhline(0, color="grey", linewidth=0.7, alpha=0.5)
ax.axvline(0, color="grey", linewidth=0.7, alpha=0.5)
ax.legend(title="Group", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()